# Tree Models

## Objectives
The objective of this notebook is to understand tree models by writing them ourselves. Sections 1 to 3 explain a single decision tree, bagging and boosting. Sections 4 and 5 write the three models by hand with numpy and check them against scikit-learn. It is an exercise in the implementation, so the models are not tuned or submitted here.

## 1. Decision Tree
A regression tree asks yes or no questions about one column at a time, and predicts the mean price of the train houses in the leaf a house ends up in. The columns and values below are made up to show the shape.

```mermaid
flowchart TD
    A["OverallQual <= 6?"] -->|Yes| B["GrLivArea <= 1500?"]
    A -->|No| C["GrLivArea <= 2000?"]
    B -->|Yes| D["Leaf: mean price"]
    B -->|No| E["Leaf: mean price"]
    C -->|Yes| F["Leaf: mean price"]
    C -->|No| G["Leaf: mean price"]
```

- The root is the top node, the branches are the nodes with a question, and the leaves are the nodes at the bottom. A yes goes left.
- A classification tree predicts the most common category in the leaf, and splits by how pure the categories are.

### Choosing a Split
At each node we try every column $j$ and every threshold $s$, and keep the pair that minimises the squared error of the two groups, each predicted by its mean:
$$\min_{j,\, s} \left[ \sum_{i \in R_L} (y_i - \bar y_L)^2 + \sum_{i \in R_R} (y_i - \bar y_R)^2 \right], \qquad R_L = \{x_j \le s\},\; R_R = \{x_j > s\}$$
The thresholds are the midpoints between the sorted values of the column. The search is greedy: it picks the best split for this node, then repeats in each group.

### Pruning
A tree grown until every leaf has one house fits the noise: a low bias and a high variance. There are two ways to control it:
- Pre-pruning stops the growth early, with a limit on the depth, on the houses per leaf, or on how much a split must reduce the error. We choose the limit by cross-validation.
- Post-pruning grows a large tree and cuts it back by minimising $\text{RSS}(T) + \alpha |T|$, where $|T|$ is the number of leaves. Raising $\alpha$ removes the branches that reduce the error least per leaf, one at a time, which gives a sequence of smaller trees. We choose $\alpha$ by cross-validation. This $\alpha$ is not the penalty of the regularisation notebook.

### Properties
- No standardising or log of the columns is needed, because a split only uses the order of the values.
- Interactions are found by themselves.
- Predictions do not extrapolate, because a house outside the train range falls in the outermost leaf.
- The variance is high, because a small change in the houses can change an early split.

## 2. Bagging
Bagging, short for bootstrap aggregating, reduces the variance of any model: fit the model to many bootstrap samples and average the predictions. It helps most for high-variance models, so it is usually applied to deep trees. The steps are:
- Draw a bootstrap sample: $n$ houses drawn with replacement from the $n$ train houses, so some appear twice and some not at all.
- Fit a model to it, here a deep tree.
- Repeat for $B$ samples. A prediction is the average of the models, or the majority vote for a classification.

### Random Forest
A random forest is bagged trees with one extra step: each split tries only a random subset of the columns.

```mermaid
flowchart LR
    D["Train houses"] --> S1["Bootstrap sample 1"]
    D --> S2["Bootstrap sample 2"]
    D --> S3["Bootstrap sample B"]
    S1 --> T1["Deep tree 1"]
    S2 --> T2["Deep tree 2"]
    S3 --> T3["Deep tree B"]
    T1 --> M["Average"]
    T2 --> M
    T3 --> M
```

If the $B$ predictions each have variance $\sigma^2$ and a correlation $\rho$ between any two, the variance of their average is
$$\rho\,\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2$$
- The second term goes to 0 as $B$ grows, so more trees never overfit.
- The first term does not, so the trees must differ from each other. The bootstrap samples make them differ. The random columns of a forest make them differ more, as otherwise the strongest columns would be the first split of nearly every tree.
- Averaging leaves the bias about the same, so the trees are grown deep.

### Out-of-Bag Error
A house is missing from a bootstrap sample with probability $(1 - 1/n)^n \approx 0.368$, so each tree misses about a third of the houses. Predicting each house with only the trees that missed it, and measuring the error, gives the out-of-bag error: an estimate for unseen houses with no separate validation set. We use it to choose the number of columns tried at each split. A common start is a third of the columns for regression.

## 3. Boosting
Boosting reduces the bias instead of the variance. It starts from a simple model, and repeatedly fits a small tree to the mistakes the model still makes, then adds a fraction of that tree to the model. For squared error the steps are:
- Start with the mean of the train prices as the prediction for every house.
- Compute the residuals, the amount by which the current model is wrong for each house.
- Fit a small tree to the residuals, so each leaf predicts the mean residual of its houses.
- Add the tree to the model after multiplying it by a learning rate $\eta$ between 0 and 1: $F_m = F_{m-1} + \eta \times \text{tree}$. Then repeat from the residuals.

The residual is the negative gradient of the squared error, so each step is a gradient descent step in the space of functions, which is why it is called gradient boosting.

For example, take four houses with targets 2, 4, 6 and 12, one-split trees and $\eta = 0.5$. The numbers are made up.
- The mean predicts 6 for all, the residuals are -4, -2, 0 and 6, and the squared error is 56.
- Tree 1 separates the first three houses from the last, with leaf values -2 and 6. The predictions become 5, 5, 5 and 9, and the error falls to 20.
- Tree 2 fits the new residuals, -3, -1, 1 and 3, with leaf values -2 and 2 for the first two houses and the last two. The predictions become 4, 4, 6 and 10, and the error falls to 8.

Each tree is too simple to fit the houses alone, but their sum can. In practice there are 100 or more trees of 3 to 6 levels. A smaller learning rate generalises better but needs more trees. Unlike a forest, boosting can overfit if it runs too long, so the number of trees is chosen by validation error. XGBoost and LightGBM are fast implementations of this idea, with extra regularisation of the trees.

## 4. Features
A tree needs no standardising, because a split only uses the order of the values, and no log of the columns for the same reason. Collinear columns are not a problem either. So the data for the checks is the output of `clean` and `engineer` as it is, with every column the linear notebooks started from, and none of the logs, squares or dropped columns. The target is $\log(\text{SalePrice})$. We ask:
- How many houses and columns are there, and are any values missing?

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from src.data import load_train
from src.engineering import engineer
from src.features import clean
from src.trees import GradientBoosting, RandomForest, RegressionTree

train = load_train()
train_clean = clean(train, train)
features = engineer(train_clean, train_clean)
X = features.to_numpy(dtype=float)
y = np.log(train_clean["SalePrice"]).to_numpy()
print("Houses and columns:", X.shape)
print("Missing values:", int(features.isna().sum().sum()))
print("Columns with fewer than 2 distinct values:", int((features.nunique() < 2).sum()))

Houses and columns: (1458, 177)
Missing values: 0
Columns with fewer than 2 distinct values: 0


## 5. Implementation
The models are in `src/trees.py`, written with numpy.
- `best_split` finds the best split of a node. For each column it sorts the houses once, and the cumulative sums of $y$ and $y^2$ give the squared error of every threshold at the same time, because the error of a group is $\sum y^2 - (\sum y)^2 / \text{count}$.
- `RegressionTree` grows the tree greedily with a limit on the depth and on the houses per leaf, and predicts by sending the houses down the tree together.
- `RandomForest` fits `RegressionTree` to bootstrap samples with random columns at each split, and keeps the out-of-bag houses of each tree.
- `GradientBoosting` starts from the mean and fits each tree to the residuals, adding it with the learning rate.

We check them against scikit-learn. A single tree and boosting are deterministic, so they should match, while a forest uses random draws that differ, so we compare its error on houses it did not see. We ask:
- Does the tree give the same predictions as `DecisionTreeRegressor` for several limits?
- Does boosting match `GradientBoostingRegressor`?
- Does the forest have the same error as `RandomForestRegressor`, and the same out-of-bag error?
- How much slower is numpy?

In [2]:
def timed(function):
    start = time.time()
    result = function()
    return result, time.time() - start


# A single tree against scikit-learn, for several limits on the depth and on the houses per leaf.
trees = {}
for depth, leaf in [(3, 1), (5, 5), (8, 10), (None, 1)]:
    mine, seconds_mine = timed(lambda: RegressionTree(depth, leaf).fit(X, y))
    ref, seconds_ref = timed(lambda: DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf, random_state=0).fit(X, y))
    trees[f"depth {depth}, leaf {leaf}"] = {
        "Largest difference": np.abs(mine.predict(X) - ref.predict(X)).max(),
        "Leaves (hand-written)": sum(child == -1 for child in mine.left),
        "Leaves (scikit-learn)": ref.get_n_leaves(),
        "Seconds (hand-written)": seconds_mine,
        "Seconds (scikit-learn)": seconds_ref,
    }
display(pd.DataFrame(trees).T)

# Boosting against scikit-learn, with 100 trees of depth 3 and a learning rate of 0.1.
mine, seconds_mine = timed(lambda: GradientBoosting(100, 0.1, 3).fit(X, y))
ref, seconds_ref = timed(lambda: GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X, y))
print(f"Boosting: largest difference {np.abs(mine.predict(X) - ref.predict(X)).max():.1e}, {seconds_mine:.1f} seconds against {seconds_ref:.1f}")

# A forest of 100 trees with a third of the columns at each split: error on 358 houses it did not see, and the out-of-bag error.
order = np.random.default_rng(0).permutation(len(y))
fit, held_out = order[:1100], order[1100:]
mine, seconds_mine = timed(lambda: RandomForest(100, X.shape[1] // 3, seed=0).fit(X[fit], y[fit]))
ref, seconds_ref = timed(lambda: RandomForestRegressor(100, max_features=1 / 3, oob_score=True, random_state=0).fit(X[fit], y[fit]))
rmse = lambda actual, predicted: np.sqrt(np.mean((actual - predicted) ** 2))
forest = pd.DataFrame(
    {
        "Held-out RMSE": [rmse(y[held_out], mine.predict(X[held_out])), rmse(y[held_out], ref.predict(X[held_out]))],
        "Out-of-bag RMSE": [rmse(y[fit], mine.out_of_bag_prediction(X[fit])), rmse(y[fit], ref.oob_prediction_)],
        "Seconds": [seconds_mine, seconds_ref],
    },
    index=["Hand-written", "scikit-learn"],
)
display(forest.round(4))

,Largest difference,Leaves (hand-written),Leaves (scikit-learn),Seconds (hand-written),Seconds (scikit-learn)
"depth 3, leaf 1",7.105427e-15,8.0,8.0,0.090141,0.009445
"depth 5, leaf 5",1.243450e-14,31.0,31.0,0.250973,0.011225
"depth 8, leaf 10",5.329071e-15,89.0,89.0,0.586623,0.014538
"depth None, leaf 1",0.000000e+00,1385.0,1412.0,8.168106,0.029665


Boosting: largest difference 5.3e-15, 7.3 seconds against 0.7


,Held-out RMSE,Out-of-bag RMSE,Seconds
Hand-written,0.1373,0.1300,132.8090
scikit-learn,0.1396,0.1295,0.6491


##### Findings:
- The tree gives the same predictions as scikit-learn, to within 1e-14, for all four settings. For the three limited trees the number of leaves is the same (8, 31 and 89).
- The fully grown tree fits the train houses exactly in both, but has 1,385 leaves against 1,412. Every leaf in both is pure, and the extra leaves in scikit-learn separate houses with the same price, which does not change a prediction. This is probably floating-point rounding, which makes scikit-learn see a small difference in a group of equal prices.
- Boosting matches `GradientBoostingRegressor` to 5e-15.
- The forests use different random draws, so they cannot match, but they are at the same level. The error on the 358 held-out houses is 0.1373 for the hand-written forest and 0.1396 for scikit-learn, and the out-of-bag error is 0.1300 and 0.1295.
- The hand-written code is slower, by about 10 times for a limited tree and for boosting (7.3 seconds against 0.7). It is about 270 times slower for the fully grown tree (8.2 seconds against 0.03), and about 200 times for the forest of 100 trees (133 seconds against 0.65), because scikit-learn is compiled.